In [1]:
import numpy as np
import plotly.graph_objects as go

In [2]:
# ============================================================
# 🎲 VARIABLE SEED: Generate different but traceable results
# ============================================================
import numpy as np
import time

# Generate a seed based on current time (microseconds)
# This ensures each execution has different results
RANDOM_SEED = int((time.time() * 1000000) % 100000)  # Seed between 0-99999
np.random.seed(RANDOM_SEED)

# Prominent message to track performance
print("🎲" + "="*60)
print(f"🎯 CURRENT SEED: {RANDOM_SEED}")
print("   ⚡ RECORD THIS SEED IF YOU GET GOOD RESULTS")
print("   🔄 To reproduce: change line 8 to RANDOM_SEED = {0}".format(RANDOM_SEED))
print("="*62)

🎲============================================================
🎯 CURRENT SEED: 10932
   ⚡ RECORD THIS SEED IF YOU GET GOOD RESULTS
   🔄 To reproduce: change line 8 to RANDOM_SEED = 10932


In [3]:
# ============================================================
# 1) True nonlinear system (Lorenz Chaotic System)
# ============================================================
def plant_dynamics(x, u, sigma=5.0, rho=14.0, beta=4.0/3.0):
    """
    Continuous dynamics for Lorenz chaotic system: x = [x1, x2, x3]. 
    Returns x_dot.
    
    The Lorenz equations:
    dx1/dt = σ(x2 - x1) + u
    dx2/dt = x1(ρ - x3) - x2
    dx3/dt = x1*x2 - β*x3
    
    where:
    x1, x2, x3 = state variables
    u = external control input applied to first equation
    σ = Prandtl number (5.0, scaled for stability)
    ρ = Rayleigh number (14.0, scaled for stability)  
    β = geometric factor (4/3, scaled for stability)
    """
    x1, x2, x3 = x
    control_input = u[0] if len(u) > 0 else 0.0  # applied control
    
    # Lorenz system dynamics
    x1_dot = sigma * (x2 - x1) + control_input
    x2_dot = x1 * (rho - x3) - x2
    x3_dot = x1 * x2 - beta * x3
    
    return np.array([x1_dot, x2_dot, x3_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='mixed', process_noise_std=0.05, 
          terrain_roughness=0.02, sensor_bias=[0.0, 0.0, 0.0]):
    """
    One Euler step of the discrete plant with realistic Lorenz system disturbances.
    
    Args:
        x_k: current state [x1, x2, x3]
        u_k: control input [force] 
        dt: time step
        process_noise_type: type of noise ('mixed', 'gaussian', 'laplacian')
        process_noise_std: standard deviation of process noise
        terrain_roughness: external disturbances
        sensor_bias: systematic biases in measurements [x1_bias, x2_bias, x3_bias]
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot
    
    # Realistic Lorenz system disturbances
    
    # 1. External disturbances (environmental perturbations)
    external_noise = terrain_roughness * np.array([
        0.1 * np.sin(0.5 * x_kp1[0]) * np.random.randn(),  # x1-dependent disturbance
        0.1 * np.cos(0.3 * x_kp1[1]) * np.random.randn(),  # x2-dependent disturbance
        0.05 * x_kp1[2] / (1 + x_kp1[2]**2) * np.random.randn()   # x3-dependent disturbance
    ])
    
    # 2. State magnitude-dependent noise (chaotic sensitivity)
    state_magnitude = np.linalg.norm(x_kp1)
    magnitude_noise_factor = 1 + 0.05 * np.tanh(state_magnitude / 20.0)
    
    # 3. Mixed process noise (combination of different noise types)
    if process_noise_type == 'mixed':
        # Gaussian component (main noise)
        gaussian_noise = np.random.normal(0, process_noise_std * magnitude_noise_factor, size=x_kp1.shape)
        # Impulse noise (occasional large disturbances)
        impulse_prob = 0.02  # 2% chance of impulse noise
        impulse_noise = np.zeros_like(x_kp1)
        if np.random.rand() < impulse_prob:
            impulse_noise = np.random.normal(0, process_noise_std * 5, size=x_kp1.shape)
        # Laplacian component (heavy-tailed noise)
        laplacian_noise = np.random.laplace(0, process_noise_std * 0.3, size=x_kp1.shape)
        
        total_noise = gaussian_noise + impulse_noise + laplacian_noise
    elif process_noise_type == 'laplacian':
        total_noise = np.random.laplace(0, process_noise_std * magnitude_noise_factor, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std * magnitude_noise_factor
        total_noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        total_noise = np.random.normal(0, process_noise_std * magnitude_noise_factor, size=x_kp1.shape)
    
    # 4. Systematic biases (drift, calibration errors)
    bias_noise = np.array(sensor_bias) * dt
    
    # 5. Sensor quantization effects
    sensor_resolution = 0.001  # 0.001 unit resolution
    quantization_noise = sensor_resolution * (np.random.rand(3) - 0.5)
    
    # Combine all disturbances
    x_kp1 += external_noise + total_noise + bias_noise + quantization_noise
    
    return x_kp1

def generate_realistic_trajectory(t, trajectory_type='chaotic_excitation'):
    """
    Generate realistic control inputs for Lorenz chaotic system.
    
    Args:
        t: time value
        trajectory_type: 'chaotic_excitation', 'stabilize', 'oscillate', 'mixed', 'resonance'
    
    Returns:
        u: [control force] control input
    """
    if trajectory_type == 'chaotic_excitation':
        # Excitation to explore chaotic dynamics (reduced for stability)
        excitation_control = 2.0 * np.sin(0.5 * t) * np.exp(-0.02 * t)
        return np.array([excitation_control])
    
    elif trajectory_type == 'stabilize':
        # Small control effort for system stabilization
        control = 0.5 * np.sin(0.1 * t)
        return np.array([control])
    
    elif trajectory_type == 'oscillate':
        # Sinusoidal forcing with multiple frequencies (reduced for stability)
        control = 1.5 * np.sin(0.8 * t) + 0.5 * np.cos(1.5 * t)
        return np.array([control])
    
    elif trajectory_type == 'resonance':
        # Resonance-like forcing for rich dynamics
        base_freq = 0.6
        resonance_control = 4.0 * np.sin(base_freq * t) * np.cos(0.3 * base_freq * t)
        return np.array([resonance_control])
    
    else:  # 'mixed' - combination of different control strategies
        # Mixed control with different phases for rich system identification
        phase = (t % 20.0) / 20.0  # 20-second cycles
        
        if phase < 0.25:  # Chaotic excitation phase (reduced amplitude)
            control = 2.0 * np.sin(0.7 * t)
            return np.array([control])
        elif phase < 0.5:  # Multi-frequency oscillation (reduced amplitude)
            control = 1.0 * np.sin(t) + 0.5 * np.cos(2.5 * t)
            return np.array([control])
        elif phase < 0.75:  # Low amplitude stabilization
            control = 0.2 * np.sin(3 * t)
            return np.array([control])
        else:  # Complex multi-harmonic forcing (reduced amplitude)
            control = 1.5 * np.sin(1.2 * t) + 0.8 * np.cos(2.8 * t) + 0.3 * np.sin(4.1 * t)
            return np.array([control])

In [4]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    # z = np.clip(z, -500, 500)  # Clipping commented out - let it handle naturally 
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est, u_input=None):
    """
    RHONN features for Lorenz chaotic system.
    For a 3-state Lorenz system: x = [x1, x2, x3], u = [force]
    
    Feature vector z = [x1, x2, x3, x1*x2, x1*x3, x2*x3, x1^2, x2^2, x3^2, u, 1]
    
    This captures essential nonlinearities for Lorenz dynamics:
    - x1, x2, x3: Direct state terms for linear dynamics
    - x1*x2, x1*x3, x2*x3: Bilinear coupling terms (essential for Lorenz)
    - x1^2, x2^2, x3^2: Quadratic terms for enhanced nonlinear modeling
    - u: Control input
    - 1: Bias term
    """
    x1 = np.clip(x_est[0], -10, 10)  # first state (aggressively clipped for stability)
    x2 = np.clip(x_est[1], -10, 10)  # second state (aggressively clipped for stability)
    x3 = np.clip(x_est[2], -10, 10)  # third state (aggressively clipped for stability)
    
    # Essential nonlinear features for Lorenz system (with aggressive stability constraints)
    features = [
        x1,                               # Direct x1 term
        x2,                               # Direct x2 term
        x3,                               # Direct x3 term
        np.clip(x1 * x2, -20, 20),       # Bilinear coupling (aggressively clipped)
        np.clip(x1 * x3, -20, 20),       # x1-x3 coupling (aggressively clipped)
        np.clip(x2 * x3, -20, 20),       # x2-x3 coupling (aggressively clipped)
        np.clip(x1 * x1, 0, 20),         # x1 squared term (aggressively clipped)
        np.clip(x2 * x2, 0, 20),         # x2 squared term (aggressively clipped)
        np.clip(x3 * x3, 0, 20),         # x3 squared term (aggressively clipped)
    ]
    
    # Add control input if available
    if u_input is not None and len(u_input) >= 1:
        features.append(u_input[0])  # Direct force input
    else:
        features.append(0.0)         # Zero control placeholder
    
    # Bias term
    features.append(1.0)
    
    return np.array(features)


def RHONN_predict(x_state_for_z, w_neuron, u_input=None):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z, u_input)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [5]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for Lorenz system)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        # For Lorenz system, use filter's own estimates for all three states
        x_state_for_z[0] = chi_k[0]  # x1 (filter's own estimate)
        x_state_for_z[1] = chi_k[1]  # x2 (filter's own estimate)  
        x_state_for_z[2] = chi_k[2]  # x3 (filter's own estimate)

        z_i = construct_z_vector(x_state_for_z, u_input)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

In [6]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=None, R_std=None, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        
        # Handle Q_std (process noise) - can be scalar or list/array per neuron
        if Q_std is None:
            self.Q_std = [0.05] * num_neurons
        elif isinstance(Q_std, (int, float)):
            self.Q_std = [Q_std] * num_neurons
        else:
            self.Q_std = list(Q_std) if len(Q_std) == num_neurons else [0.05] * num_neurons
            
        # Handle R_std (measurement noise) - can be scalar or list/array per neuron  
        if R_std is None:
            self.R_std = [0.1] * num_neurons
        elif isinstance(R_std, (int, float)):
            self.R_std = [R_std] * num_neurons
        else:
            self.R_std = list(R_std) if len(R_std) == num_neurons else [0.1] * num_neurons
            
        # Compute R_var for each neuron
        self.R_var = [r**2 for r in self.R_std]
        
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        # Initialize global weight estimates first
        self.weights = []
        if initial_weights is not None:
            self.weights = [np.copy(w) for w in initial_weights]
        else:
            self.weights = [np.random.randn(num_weights_per_neuron) * 0.01 for _ in range(num_neurons)]

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                # Adaptive initialization variance based on weight magnitudes
                weight_magnitude = np.std(base) if np.std(base) > 0 else 1.0
                init_std = max(0.01, min(0.1, weight_magnitude * 0.5))  # Adaptive but bounded
            else:
                base = self.weights[i]
                init_std = 0.05
            
            # Better initialization: base + controlled noise
            particles_i = base[np.newaxis, :] + np.random.randn(n_particles, num_weights_per_neuron) * init_std
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        """Effective Sample Size calculation with improved numerical stability."""
        w_norm = w / (np.sum(w) + 1e-15)
        return 1.0 / (np.sum(w_norm**2) + 1e-15)

    def _resample_stratified(self, neuron_index):
        """Stratified resampling (reduces variance compared to systematic/multinomial)"""
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w_norm = w / (np.sum(w) + 1e-15)
        N = len(w_norm)
        cdf = np.cumsum(w_norm)
        
        # Stratified positions: divide [0,1] into N strata
        positions = (np.random.rand(N) + np.arange(N)) / N
        
        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N and j < N:
            if positions[i] <= cdf[j]:
                indexes[i] = j
                i += 1
            else:
                j += 1
        
        # Handle any remaining indices
        while i < N:
            indexes[i] = N - 1
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : filter's own estimate at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        u_input: control input at time k (for mobile robot)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # theta angle (filter's own estimate for pendulum)
        z = construct_z_vector(x_state_for_z, u_input)  # (num_features,)

        # 1) Predict: random walk on weights with state-specific Q_std
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std[i]

        # 2) Update: importance weights with improved Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Improved log-likelihood calculation with better numerical stability
            var_robust = max(self.R_var[i], 1e-6)  # Avoid division by very small numbers
            ll = -0.5 * (innov**2) / var_robust - 0.5 * np.log(2 * np.pi * var_robust)
            
            # Normalize for numerical stability
            ll_max = np.max(ll)
            ll_normalized = ll - ll_max
            like = np.exp(np.clip(ll_normalized, -20, 0))  # Clip to avoid underflow

            # Update weights with better safeguards
            self.weights_pf[i] *= (like + 1e-15)
            w_sum = np.sum(self.weights_pf[i])
            
            if w_sum < 1e-15:
                # Complete weight collapse - reinitialize uniformly
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= w_sum

            # 3) Resample if ESS is low (using improved stratified resampling)
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_stratified(i)

        # 4) Update global weight estimates (weighted mean of particles for consistency)
        if not hasattr(self, 'weights'):
            self.weights = []
        
        # Ensure we have the right number of weight vectors
        while len(self.weights) < self.num_neurons:
            self.weights.append(np.zeros(self.num_weights_per_neuron))
            
        for i in range(self.num_neurons):
            w_norm = self.weights_pf[i] / (np.sum(self.weights_pf[i]) + 1e-15)
            self.weights[i] = np.sum(w_norm[:, np.newaxis] * self.particles[i], axis=0)

    def get_estimate(self):
        """Return current weight estimates (maintained consistently with particles)."""
        if hasattr(self, 'weights') and len(self.weights) == self.num_neurons:
            return self.weights
        else:
            # Fallback to simple mean if weights not properly maintained
            return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]

    def get_parameters_info(self):
        """Return comprehensive information about the PF parameters and state for each neuron."""
        state_names = ['x', 'y', 'theta']
        info = {}
        for i in range(self.num_neurons):
            name = state_names[i] if i < len(state_names) else f'state_{i}'
            current_ess = self._ess(self.weights_pf[i]) if hasattr(self, 'weights_pf') else 'N/A'
            info[name] = {
                'Q_std': self.Q_std[i],
                'R_std': self.R_std[i], 
                'R_var': self.R_var[i],
                'n_particles': self.n_particles,
                'ess_threshold': self.ess_threshold,
                'current_ess': current_ess,
                'ess_ratio': current_ess / self.n_particles if isinstance(current_ess, (int, float)) else 'N/A'
            }
        return info

In [7]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        # Calculate lambda and weights for sigma points
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Weights for mean and covariance computation
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        """Generate sigma points for UKF."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, filter's own estimate at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        u_input: control input at time k (for mobile robot)
        """
        # Build series-parallel state for z: use filter's own estimate at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # theta angle (filter's own estimate for pendulum)

        z_i = construct_z_vector(x_state_for_z, u_input)  # shape (num_features,)

        for i in range(self.num_neurons):
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

In [8]:
# ============================================================
# 4c) Rao-Blackwellized Particle Filter (RBPF) trainer over weights
# ============================================================
class RBPF_RHONN_Trainer:
    """
    Rao-Blackwellized Particle Filter on each neuron's weight vector.
    
    The RBPF combines:
    1. Particle filtering for nonlinear parameters (subset of weights)
    2. Kalman filtering for linear parameters (remaining weights)
    
    This approach is more efficient than pure PF by exploiting linearity
    where possible in the weight update equations.
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 n_particles=100, linear_dims_ratio=0.5, Q_std=None, R_std=None, 
                 ess_threshold=None, Q_linear=1e-4, R_linear=1e-2):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        
        # Determine which dimensions are treated as linear vs nonlinear
        self.n_linear = int(num_weights_per_neuron * linear_dims_ratio)
        self.n_nonlinear = num_weights_per_neuron - self.n_linear
        
        # Linear dimensions: typically bias terms and direct state terms (last few components)
        self.linear_indices = list(range(num_weights_per_neuron - self.n_linear, num_weights_per_neuron))
        # Nonlinear dimensions: typically sigmoid and trigonometric features (first components)
        self.nonlinear_indices = list(range(self.n_nonlinear))
        
        print(f"RBPF Configuration:")
        print(f"  Linear dims ({self.n_linear}): {self.linear_indices}")
        print(f"  Nonlinear dims ({self.n_nonlinear}): {self.nonlinear_indices}")
        
        # Handle Q_std for nonlinear parameters
        if Q_std is None:
            self.Q_std = [0.05] * num_neurons
        elif isinstance(Q_std, (int, float)):
            self.Q_std = [Q_std] * num_neurons
        else:
            self.Q_std = list(Q_std) if len(Q_std) == num_neurons else [0.05] * num_neurons
            
        # Handle R_std for measurements
        if R_std is None:
            self.R_std = [0.1] * num_neurons
        elif isinstance(R_std, (int, float)):
            self.R_std = [R_std] * num_neurons
        else:
            self.R_std = list(R_std) if len(R_std) == num_neurons else [0.1] * num_neurons
            
        self.R_var = [r**2 for r in self.R_std]
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0
        
        # Initialize weights
        self.weights = []
        if initial_weights is not None:
            self.weights = [np.copy(w) for w in initial_weights]
        else:
            self.weights = [np.random.randn(num_weights_per_neuron) * 0.01 for _ in range(num_neurons)]
        
        # Particle filter components (for nonlinear parameters)
        self.particles_nonlinear = []  # Particles for nonlinear weight components
        self.weights_pf = []  # Particle weights
        
        # Kalman filter components (for linear parameters given particles)
        self.mean_linear = []  # Conditional mean of linear weights
        self.cov_linear = []   # Conditional covariance of linear weights
        
        self.Q_linear = Q_linear  # Process noise for linear parameters
        self.R_linear = R_linear  # Measurement noise for linear parameters
        
        # Initialize for each neuron
        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base_weights = np.copy(initial_weights[i])
            else:
                base_weights = self.weights[i]
            
            # Initialize nonlinear particles
            nonlinear_base = base_weights[self.nonlinear_indices]
            weight_magnitude = np.std(nonlinear_base) if np.std(nonlinear_base) > 0 else 1.0
            init_std = max(0.01, min(0.1, weight_magnitude * 0.3))
            
            particles_nl = nonlinear_base[np.newaxis, :] + np.random.randn(n_particles, self.n_nonlinear) * init_std
            self.particles_nonlinear.append(particles_nl)
            self.weights_pf.append(np.ones(n_particles) / n_particles)
            
            # Initialize linear Kalman filter components
            linear_base = base_weights[self.linear_indices]
            self.mean_linear.append([linear_base.copy() for _ in range(n_particles)])
            self.cov_linear.append([np.eye(self.n_linear) * 1.0 for _ in range(n_particles)])
    
    def _ess(self, w):
        """Effective Sample Size calculation."""
        w_norm = w / (np.sum(w) + 1e-15)
        return 1.0 / (np.sum(w_norm**2) + 1e-15)
    
    def _resample_stratified(self, neuron_index):
        """Stratified resampling for both nonlinear particles and linear Kalman filters."""
        w = self.weights_pf[neuron_index]
        w_norm = w / (np.sum(w) + 1e-15)
        N = len(w_norm)
        cdf = np.cumsum(w_norm)
        
        # Stratified positions
        positions = (np.random.rand(N) + np.arange(N)) / N
        
        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N and j < N:
            if positions[i] <= cdf[j]:
                indexes[i] = j
                i += 1
            else:
                j += 1
        
        while i < N:
            indexes[i] = N - 1
            i += 1
        
        # Resample nonlinear particles
        self.particles_nonlinear[neuron_index] = self.particles_nonlinear[neuron_index][indexes]
        
        # Resample linear Kalman filter states
        new_mean_linear = [self.mean_linear[neuron_index][idx].copy() for idx in indexes]
        new_cov_linear = [self.cov_linear[neuron_index][idx].copy() for idx in indexes]
        self.mean_linear[neuron_index] = new_mean_linear
        self.cov_linear[neuron_index] = new_cov_linear
        
        # Reset particle weights
        self.weights_pf[neuron_index] = np.ones(N) / N
    
    def _construct_z_linear_nonlinear(self, x_state, u_input, nonlinear_weights):
        """
        Construct feature vector split into linear and nonlinear parts.
        Given the nonlinear weights, compute the contribution and build H matrix for linear part.
        """
        z_full = construct_z_vector(x_state, u_input)
        
        # Split into nonlinear and linear components
        z_nonlinear = z_full[self.nonlinear_indices]
        z_linear = z_full[self.linear_indices]
        
        # Nonlinear contribution (already determined by particles)
        nonlinear_contribution = np.dot(nonlinear_weights, z_nonlinear)
        
        return z_linear, nonlinear_contribution, z_full
    
    def update(self, chi_kp1, chi_k, x_hat_previous, u_input=None):
        """
        RBPF update combining particle filter for nonlinear weights 
        and Kalman filter for linear weights.
        """
        # Build state for feature construction
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # theta angle (filter's estimate)
        
        for i in range(self.num_neurons):
            target = chi_kp1[i]
            
            # 1) Predict nonlinear particles (random walk)
            self.particles_nonlinear[i] += np.random.randn(self.n_particles, self.n_nonlinear) * self.Q_std[i]
            
            # 2) For each particle, update corresponding linear Kalman filter
            log_likelihoods = np.zeros(self.n_particles)
            
            for p in range(self.n_particles):
                nonlinear_weights_p = self.particles_nonlinear[i][p]
                
                # Get linear observation model
                z_linear, nonlinear_contrib, _ = self._construct_z_linear_nonlinear(
                    x_state_for_z, u_input, nonlinear_weights_p)
                
                # Kalman predict step for linear weights
                # Mean remains the same (random walk model)
                mean_pred = self.mean_linear[i][p].copy()
                cov_pred = self.cov_linear[i][p] + np.eye(self.n_linear) * self.Q_linear
                
                # Kalman update step
                H = z_linear.reshape(1, -1)  # Observation matrix (1 x n_linear)
                
                # Innovation covariance
                S = H @ cov_pred @ H.T + self.R_linear
                S_scalar = S[0, 0] if S.shape == (1, 1) else np.trace(S)
                
                if S_scalar < 1e-12:
                    S_scalar = 1e-12
                
                # Kalman gain
                K = cov_pred @ H.T / S_scalar  # (n_linear x 1)
                
                # Predicted observation (including nonlinear contribution)
                y_pred = H @ mean_pred + nonlinear_contrib
                
                # Innovation
                innovation = target - y_pred[0] if hasattr(y_pred, '__len__') else target - y_pred
                
                # Update linear weights for this particle
                self.mean_linear[i][p] = mean_pred + K.flatten() * innovation
                self.cov_linear[i][p] = (np.eye(self.n_linear) - np.outer(K.flatten(), H.flatten())) @ cov_pred
                
                # Ensure positive definiteness
                self.cov_linear[i][p] = 0.5 * (self.cov_linear[i][p] + self.cov_linear[i][p].T)
                eigenvals = np.linalg.eigvals(self.cov_linear[i][p])
                if np.min(eigenvals) <= 0:
                    self.cov_linear[i][p] += np.eye(self.n_linear) * 1e-6
                
                # Compute log-likelihood for particle weight update
                log_likelihoods[p] = -0.5 * (innovation**2) / S_scalar - 0.5 * np.log(2 * np.pi * S_scalar)
            
            # 3) Update particle weights based on likelihoods
            # Normalize log-likelihoods for numerical stability
            ll_max = np.max(log_likelihoods)
            ll_normalized = log_likelihoods - ll_max
            likelihoods = np.exp(np.clip(ll_normalized, -20, 0))
            
            # Update particle weights
            self.weights_pf[i] *= (likelihoods + 1e-15)
            w_sum = np.sum(self.weights_pf[i])
            
            if w_sum < 1e-15:
                # Weight collapse - reinitialize uniformly
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= w_sum
            
            # 4) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_stratified(i)
    
    def get_estimate(self):
        """Return current weight estimates by combining nonlinear particles and linear means."""
        estimates = []
        
        for i in range(self.num_neurons):
            # Weighted combination of particles
            w_norm = self.weights_pf[i] / (np.sum(self.weights_pf[i]) + 1e-15)
            
            # Combine nonlinear and linear estimates
            full_weights = np.zeros(self.num_weights_per_neuron)
            
            # Nonlinear part: weighted average of particles
            nonlinear_est = np.sum(w_norm[:, np.newaxis] * self.particles_nonlinear[i], axis=0)
            full_weights[self.nonlinear_indices] = nonlinear_est
            
            # Linear part: weighted average of Kalman means
            linear_means = np.array([self.mean_linear[i][p] for p in range(self.n_particles)])
            linear_est = np.sum(w_norm[:, np.newaxis] * linear_means, axis=0)
            full_weights[self.linear_indices] = linear_est
            
            estimates.append(full_weights)
        
        return estimates
    
    def get_parameters_info(self):
        """Return comprehensive information about RBPF parameters."""
        state_names = ['theta', 'omega']
        info = {}
        for i in range(min(self.num_neurons, len(state_names))):
            name = state_names[i]
            current_ess = self._ess(self.weights_pf[i]) if hasattr(self, 'weights_pf') else 'N/A'
            info[name] = {
                'Q_std': self.Q_std[i],
                'R_std': self.R_std[i],
                'n_particles': self.n_particles,
                'n_linear': self.n_linear,
                'n_nonlinear': self.n_nonlinear,
                'ess_threshold': self.ess_threshold,
                'current_ess': current_ess,
                'ess_ratio': current_ess / self.n_particles if isinstance(current_ess, (int, float)) else 'N/A',
                'Q_linear': self.Q_linear,
                'R_linear': self.R_linear
            }
        return info

In [9]:
import math, time

# ============================================================
# Differential Evolution Optimizer (lightweight)
# ============================================================
def differential_evolution(objective, bounds, pop_factor=10, F=0.7, CR=0.9, generations=30, seed=None, tol=1e-6, stall_generations=8):
    if seed is not None:
        np.random.seed(seed)
    dim = len(bounds)
    pop_size = max(pop_factor * dim, 4)
    # Initialize population uniformly inside bounds
    pop = np.array([
        [np.random.uniform(low, high) for (low, high) in bounds]
        for _ in range(pop_size)
    ])
    scores = np.array([objective(ind) for ind in pop])
    best_idx = int(np.argmin(scores))
    best = pop[best_idx].copy()
    best_score = scores[best_idx]
    no_improve = 0
    history = [(0, best_score)]

    for gen in range(1, generations+1):
        for i in range(pop_size):
            # Mutation: select 3 distinct other indices
            idxs = [idx for idx in range(pop_size) if idx != i]
            a, b, c = pop[np.random.choice(idxs, 3, replace=False)]
            mutant = a + F * (b - c)
            # Crossover
            trial = pop[i].copy()
            j_rand = np.random.randint(0, dim)
            for j in range(dim):
                if np.random.rand() < CR or j == j_rand:
                    trial[j] = mutant[j]
            # Clamp to bounds
            for j, (low, high) in enumerate(bounds):
                if trial[j] < low: trial[j] = low
                if trial[j] > high: trial[j] = high
            # Evaluate
            trial_score = objective(trial)
            if trial_score < scores[i]:
                pop[i] = trial
                scores[i] = trial_score
                if trial_score < best_score - tol:
                    best_score = trial_score
                    best = trial.copy()
        if best_score < history[-1][1] - tol:
            no_improve = 0
        else:
            no_improve += 1
        history.append((gen, best_score))
        if no_improve >= stall_generations:
            break
    return {'best_params': best, 'best_score': best_score, 'history': history}

# ============================================================
# Objective helpers: short-horizon simulation for each filter
# ============================================================

def short_sim_prepare(common_initial_weights, num_neurons, num_features):
    # Provide fresh copies of initial weights per optimization call
    return [np.copy(w) for w in common_initial_weights]

SHORT_STEPS = 400  # reduced horizon for speed

# Control schedule reused (figure8 default). We'll precompute controls for speed.
precomputed_u = None

def ensure_precomputed_controls(dt, trajectory_type='chaotic_excitation'):
    global precomputed_u
    if precomputed_u is None or len(precomputed_u) != SHORT_STEPS:
        precomputed_u = []
        for k in range(SHORT_STEPS):
            t_current = k * dt
            precomputed_u.append(generate_realistic_trajectory(t_current, trajectory_type))
        precomputed_u = np.array(precomputed_u)
    return precomputed_u

# Shared small plant wrapper for objective

def run_short_sim_EKF(params, common_initial_weights, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias):
    Q_init, R_init, P_init, eta = params
    num_neurons = 3
    num_features = len(common_initial_weights[0])
    weights_init = short_sim_prepare(common_initial_weights, num_neurons, num_features)
    ekf_local = EKF_RHONN_Trainer(num_neurons, num_features, initial_weights=weights_init, Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta)
    x_true = np.zeros((SHORT_STEPS, 3))
    x_hat = np.zeros((SHORT_STEPS, 3))
    # initial state: Lorenz system near attractor
    x_true[0] = [1.0, 1.0, 1.0]  # initial condition in Lorenz attractor region
    x_hat[0] = x_true[0]
    controls = ensure_precomputed_controls(dt, 'swing_up')
    for k in range(SHORT_STEPS-1):
        u_k = controls[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
        ekf_local.update(chi_kp1=x_true[k+1], chi_k=x_hat[k], x_hat_previous=x_hat[k], u_input=u_k)
        x_state_z = np.copy(x_hat[k])
        x_hat[k+1, 0] = RHONN_predict(x_state_z, ekf_local.weights[0], u_k)
        x_hat[k+1, 1] = RHONN_predict(x_state_z, ekf_local.weights[1], u_k)
        x_hat[k+1, 2] = RHONN_predict(x_state_z, ekf_local.weights[2], u_k)
    # total MSE
    err = x_true - x_hat
    return np.mean(err[:,0]**2)+np.mean(err[:,1]**2)+np.mean(err[:,2]**2)


def run_short_sim_UKF(params, common_initial_weights, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias):
    Q_init, R_init, P_init, eta, alpha = params
    num_neurons = 3
    num_features = len(common_initial_weights[0])
    weights_init = short_sim_prepare(common_initial_weights, num_neurons, num_features)
    ukf_local = UKF_RHONN_Trainer(num_neurons, num_features, initial_weights=weights_init, Q_init=Q_init, R_init=R_init, P_init=P_init, eta=eta, alpha=alpha, beta=2.0)
    x_true = np.zeros((SHORT_STEPS, 3))
    x_hat = np.zeros((SHORT_STEPS, 3))
    # initial state: Lorenz system near attractor
    x_true[0] = [1.0, 1.0, 1.0]  # initial condition in Lorenz attractor region
    x_hat[0] = x_true[0]
    controls = ensure_precomputed_controls(dt, 'swing_up')
    for k in range(SHORT_STEPS-1):
        u_k = controls[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
        ukf_local.update(chi_kp1=x_true[k+1], chi_k=x_hat[k], x_hat_previous=x_hat[k], u_input=u_k)
        x_state_z = np.copy(x_hat[k])
        x_hat[k+1, 0] = RHONN_predict(x_state_z, ukf_local.weights[0], u_k)
        x_hat[k+1, 1] = RHONN_predict(x_state_z, ukf_local.weights[1], u_k)
    err = x_true - x_hat
    return np.mean(err[:,0]**2)+np.mean(err[:,1]**2)


def run_short_sim_PF(params, common_initial_weights, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias, n_particles_fixed=500):
    Q_theta, Q_omega, R_theta, R_omega, ess_ratio = params
    num_neurons = 2
    num_features = len(common_initial_weights[0])
    weights_init = short_sim_prepare(common_initial_weights, num_neurons, num_features)
    pf_local = PF_RHONN_Trainer(num_neurons, num_features, n_particles=n_particles_fixed, initial_weights=weights_init, Q_std=[Q_theta,Q_omega], R_std=[R_theta,R_omega], ess_threshold=n_particles_fixed*ess_ratio)
    # Force identical initialization
    for i in range(num_neurons):
        pf_local.particles[i] = np.tile(weights_init[i], (pf_local.n_particles,1))
        pf_local.weights_pf[i] = np.ones(pf_local.n_particles)/pf_local.n_particles
    x_true = np.zeros((SHORT_STEPS, 2))
    x_hat = np.zeros((SHORT_STEPS, 2))
    # initial state: pendulum at small angle
    x_true[0] = [0.1, 0.0]  # small initial angle
    x_hat[0] = x_true[0]
    controls = ensure_precomputed_controls(dt, 'swing_up')
    for k in range(SHORT_STEPS-1):
        u_k = controls[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
        pf_local.update(chi_kp1=x_true[k+1], chi_k=x_hat[k], x_hat_previous=x_hat[k], u_input=u_k)
        pf_w = pf_local.get_estimate()
        x_state_z = np.copy(x_hat[k])
        x_hat[k+1, 0] = RHONN_predict(x_state_z, pf_w[0], u_k)
        x_hat[k+1, 1] = RHONN_predict(x_state_z, pf_w[1], u_k)
    err = x_true - x_hat
    return np.mean(err[:,0]**2)+np.mean(err[:,1]**2)


def run_short_sim_RBPF(params, common_initial_weights, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias, n_particles_fixed=300):
    Q_theta, Q_omega, R_theta, R_omega, ess_ratio, linear_ratio = params
    num_neurons = 2
    num_features = len(common_initial_weights[0])
    weights_init = short_sim_prepare(common_initial_weights, num_neurons, num_features)
    rbpf_local = RBPF_RHONN_Trainer(num_neurons, num_features, n_particles=n_particles_fixed, 
                                   initial_weights=weights_init, linear_dims_ratio=linear_ratio,
                                   Q_std=[Q_theta,Q_omega], R_std=[R_theta,R_omega], 
                                   ess_threshold=n_particles_fixed*ess_ratio)
    x_true = np.zeros((SHORT_STEPS, 2))
    x_hat = np.zeros((SHORT_STEPS, 2))
    # initial state: pendulum at small angle
    x_true[0] = [0.1, 0.0]  # small initial angle
    x_hat[0] = x_true[0]
    controls = ensure_precomputed_controls(dt, 'swing_up')
    for k in range(SHORT_STEPS-1):
        u_k = controls[k]
        x_true[k+1] = plant(x_true[k], u_k, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
        rbpf_local.update(chi_kp1=x_true[k+1], chi_k=x_hat[k], x_hat_previous=x_hat[k], u_input=u_k)
        rbpf_w = rbpf_local.get_estimate()
        x_state_z = np.copy(x_hat[k])
        x_hat[k+1, 0] = RHONN_predict(x_state_z, rbpf_w[0], u_k)
        x_hat[k+1, 1] = RHONN_predict(x_state_z, rbpf_w[1], u_k)
    err = x_true - x_hat
    return np.mean(err[:,0]**2)+np.mean(err[:,1]**2)

# ============================================================
# Wrapper objectives with logging & penalty for instability
# ============================================================

def make_objective(filter_name, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias):
    def obj(params):
        try:
            if filter_name=='EKF':
                return run_short_sim_EKF(params, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias)
            elif filter_name=='UKF':
                return run_short_sim_UKF(params, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias)
            elif filter_name=='RBPF':
                return run_short_sim_RBPF(params, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias)
            else:  # PF
                pass
                # return run_short_sim_PF(params, common_initial_weights, dt, proc_type, proc_std, terr_rough, bias)
        except Exception as e:
            # Heavy penalty if something blows up
            return 1e6
    return obj

# ============================================================
# Launch optimization (set RUN_DE=True to execute)
# ============================================================
RUN_DE = False  # toggle to False to skip

# Define common_initial_weights before use
import numpy as np
num_neurons = 2
num_features = 7  # Simplified RHONN: [θ, ω, sin(θ), cos(θ), θω, u, 1]
num_weights_per_neuron = num_features  # Each neuron has same number of weights as features
common_initial_weights = [np.zeros(num_features) for _ in range(num_neurons)]

# ============================================================
# OPTIMIZATION COMMENTED OUT - USING MANUAL VALUES
# ============================================================
print("\n[MANUAL] Using manually tuned parameters (optimization skipped)")

# Manual parameter sets adapted for Lorenz chaotic system
# These are conservative values for numerical stability

# EKF parameters: [Q_init, R_init, P_init, eta] - more conservative for Lorenz
ekf_manual_params = [1e-5, 1e-3, 0.1, 0.1]

# UKF parameters: [Q_init, R_init, P_init, eta, alpha] - more conservative for Lorenz
ukf_manual_params = [1e-5, 1e-3, 0.1, 0.1, 1e-3]

# PF parameters: [Q_x1, Q_x2, Q_x3, R_x1, R_x2, R_x3, ess_ratio]
pf_manual_params = [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.5]

# RBPF parameters: [Q_x1, Q_x2, Q_x3, R_x1, R_x2, R_x3, ess_ratio, linear_dims_ratio]
# With 11 features, linear ratio of 0.36 gives 4 linear dims: [7,8,9,10] (x1²,x2²,x3²,u,1)
rbpf_manual_params = [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.65, 0.36]

# Create result dictionaries for compatibility with existing code
ekf_res = {'best_params': ekf_manual_params, 'best_score': 0.001}
ukf_res = {'best_params': ukf_manual_params, 'best_score': 0.001}  
pf_res = {'best_params': pf_manual_params, 'best_score': 0.001}
rbpf_res = {'best_params': rbpf_manual_params, 'best_score': 0.001}

print(f"[MANUAL][EKF] Q={ekf_manual_params[0]:.3e} R={ekf_manual_params[1]:.3e} P={ekf_manual_params[2]:.1f} eta={ekf_manual_params[3]:.2f}")
print(f"[MANUAL][UKF] Q={ukf_manual_params[0]:.3e} R={ukf_manual_params[1]:.3e} P={ukf_manual_params[2]:.1f} eta={ukf_manual_params[3]:.2f} alpha={ukf_manual_params[4]:.3e}")  
print(f"[MANUAL][PF ] Q=[{pf_manual_params[0]:.3f},{pf_manual_params[1]:.3f},{pf_manual_params[2]:.3f}] R=[{pf_manual_params[3]:.3f},{pf_manual_params[4]:.3f},{pf_manual_params[5]:.3f}] ESS_ratio={pf_manual_params[6]:.2f}")
print(f"[MANUAL][RBPF] Q=[{rbpf_manual_params[0]:.3f},{rbpf_manual_params[1]:.3f},{rbpf_manual_params[2]:.3f}] R=[{rbpf_manual_params[3]:.3f},{rbpf_manual_params[4]:.3f},{rbpf_manual_params[5]:.3f}] ESS_ratio={rbpf_manual_params[6]:.2f} Linear_ratio={rbpf_manual_params[7]:.2f}")

# COMMENTED OUT OPTIMIZATION CODE:
# if RUN_DE:
#     print("\n[DE] Starting hyperparameter optimization (short horizon)...")
#     start_total = time.time()
#     
#     # 🎲 Generate DE seed derived from main seed but different for each run
#     DE_SEED = (RANDOM_SEED + 12345) % 100000  
#     print(f"🔧 DE optimization seed: {DE_SEED} (derived from main seed: {RANDOM_SEED})")
# 
#     # Define missing variables with example/default values
#     dt = 0.05  # time step (seconds)
#     process_noise_type = 'laplacian'  
#     process_noise_std = 0.01  
#     terrain_roughness = 0.0  
#     sensor_bias = 0.0  
# 
#     # Capture current initial weights snapshot for reproducibility
#     initial_weights_snapshot = [np.copy(w) for w in common_initial_weights]
# 
#     # EKF
#     ekf_bounds = [ (1e-6,1e-2), (1e-5,1e-1), (0.1,10.0), (0.1,1.2) ]
#     ekf_obj = make_objective('EKF', initial_weights_snapshot, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
#     ekf_res = differential_evolution(ekf_obj, ekf_bounds, pop_factor=10, generations=35, seed=DE_SEED)
#     print(f"[DE][EKF] Best score={ekf_res['best_score']:.6e} params={ekf_res['best_params']}")
# 
#     # UKF - Same seed for consistency within this run
#     ukf_bounds = [ (1e-6,1e-2), (1e-5,1e-1), (0.1,10.0), (0.1,1.2), (1e-4,0.5) ]
#     ukf_obj = make_objective('UKF', initial_weights_snapshot, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
#     ukf_res = differential_evolution(ukf_obj, ukf_bounds, pop_factor=10, generations=35, seed=DE_SEED)
#     print(f"[DE][UKF] Best score={ukf_res['best_score']:.6e} params={ukf_res['best_params']}")
# 
#     # PF - Same seed for consistency within this run (fixed n_particles=500)
#     pf_bounds = [ (1e-3,1.0), (1e-3,1.0), (1e-3,1.0), (1e-3,1.0), (0.3,0.9) ]
#     pf_obj = make_objective('PF', initial_weights_snapshot, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)
#     pf_res = differential_evolution(pf_obj, pf_bounds, pop_factor=12, generations=40, seed=DE_SEED)
#     print(f"[DE][PF ] Best score={pf_res['best_score']:.6e} params={pf_res['best_params']}")
# 
#     total_time = time.time()-start_total
#     print(f"[DE] Optimization finished in {total_time:.1f}s")
# else:
#     ekf_res = ukf_res = pf_res = None

# Store for later use by main simulation rerun
optimized_params = {
    'EKF': ekf_res['best_params'] if ekf_res else None,
    'UKF': ukf_res['best_params'] if ukf_res else None,
    'PF' : pf_res['best_params'] if pf_res else None,
    'RBPF': rbpf_res['best_params'] if rbpf_res else None
}
print("Parameter sets ready for simulation:", optimized_params)


[MANUAL] Using manually tuned parameters (optimization skipped)
[MANUAL][EKF] Q=1.000e-05 R=1.000e-03 P=0.1 eta=0.10
[MANUAL][UKF] Q=1.000e-05 R=1.000e-03 P=0.1 eta=0.10 alpha=1.000e-03
[MANUAL][PF ] Q=[0.010,0.010,0.010] R=[0.010,0.010,0.010] ESS_ratio=0.50
[MANUAL][RBPF] Q=[0.010,0.010,0.010] R=[0.010,0.010,0.010] ESS_ratio=0.65 Linear_ratio=0.36
Parameter sets ready for simulation: {'EKF': [1e-05, 0.001, 0.1, 0.1], 'UKF': [1e-05, 0.001, 0.1, 0.1, 0.001], 'PF': [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.5], 'RBPF': [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.65, 0.36]}


In [10]:
# ============================================================
# 5) Simulation Main Loop (uses optimized params if present)
# ============================================================

# --- Simulation settings ---
n_steps = 1500
dt = 0.005  # Smaller time step for Lorenz system stability
t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

process_noise_type = 'gaussian'  # 'mixed' | 'laplacian' | 'uniform' | 'gaussian'
process_noise_std = 0.001  # Much smaller noise for chaotic system
terrain_roughness = 0.001  # Reduced external disturbances
sensor_bias = [0.001, 0.001, 0.001]  # Small systematic biases [x1, x2, x3]

# --- True system init ---
x_true = np.zeros((n_steps, 3))
x_true[0] = [0.1, 0.1, 0.1]  # Initial conditions for Lorenz system [x1, x2, x3] (scaled down for stability)

# --- Control trajectory ---
trajectory_type = 'mixed'  # 'chaotic_excitation', 'stabilize', 'oscillate', 'mixed', 'resonance'

# --- RHONN config ---
num_neurons = 3  # Three states for Lorenz system [x1, x2, x3]
num_features = 11  # Feature vector: [x1, x2, x3, x1*x2, x1*x3, x2*x3, x1^2, x2^2, x3^2, u, 1]
num_weights_per_neuron = num_features

# --- Common initial weights for fair comparison ---
common_initial_weights = [np.random.uniform(-0.5, 0.5, num_features) for _ in range(num_neurons)]
print("Common Initial Weights:")
for i, w in enumerate(common_initial_weights):
    print(f"  Neuron {i}: {w}")

# Extract optimized parameters if available
opt_EKF = optimized_params.get('EKF') if 'optimized_params' in globals() else None
opt_UKF = optimized_params.get('UKF') if 'optimized_params' in globals() else None
opt_PF  = optimized_params.get('PF')  if 'optimized_params' in globals() else None
opt_RBPF = optimized_params.get('RBPF') if 'optimized_params' in globals() else None

# Fallback defaults - adapted for Lorenz system
if opt_EKF is None:
    opt_EKF = [1e-5, 1e-3, 0.1, 0.1]  # More conservative for Lorenz
if opt_UKF is None:
    opt_UKF = [1e-5, 1e-3, 0.1, 0.1, 1e-3]  # More conservative for Lorenz
if opt_PF is None:
    # Map to Q_x1,Q_x2,Q_x3,R_x1,R_x2,R_x3,ess_ratio
    opt_PF = [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.5]
if opt_RBPF is None:
    # Map to Q_x1,Q_x2,Q_x3,R_x1,R_x2,R_x3,ess_ratio,linear_ratio
    opt_RBPF = [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.65, 0.4]

print("\nUsing parameter sets:")
print(f"EKF  -> Q_init={opt_EKF[0]:.3e} R_init={opt_EKF[1]:.3e} P_init={opt_EKF[2]:.3f} eta={opt_EKF[3]:.3f}")
print(f"UKF  -> Q_init={opt_UKF[0]:.3e} R_init={opt_UKF[1]:.3e} P_init={opt_UKF[2]:.3f} eta={opt_UKF[3]:.3f} alpha={opt_UKF[4]:.3e}")
print(f"PF   -> Q=[{opt_PF[0]:.3f},{opt_PF[1]:.3f}] R=[{opt_PF[2]:.3f},{opt_PF[3]:.3f}] ESS_ratio={opt_PF[4]:.2f}")
print(f"RBPF -> Q=[{opt_RBPF[0]:.3f},{opt_RBPF[1]:.3f}] R=[{opt_RBPF[2]:.3f},{opt_RBPF[3]:.3f}] ESS_ratio={opt_RBPF[4]:.2f} Linear_ratio={opt_RBPF[5]:.2f}")

# --- EKF ---
ekf_trainer = EKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=opt_EKF[0], R_init=opt_EKF[1], P_init=opt_EKF[2], eta=opt_EKF[3]
)
x_hat_ekf = np.zeros((n_steps, 3))
x_hat_ekf[0] = x_true[0]

# --- UKF ---
ukf_trainer = UKF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    initial_weights=common_initial_weights,
    Q_init=opt_UKF[0], R_init=opt_UKF[1], P_init=opt_UKF[2], eta=opt_UKF[3],
    alpha=opt_UKF[4], beta=2.0
)
x_hat_ukf = np.zeros((n_steps, 3))
x_hat_ukf[0] = x_true[0]

# --- PF --- (n_particles fixed at 800)
n_particles = 200

# Q_std_per_state = opt_PF[0:2]
# R_std_per_state = opt_PF[2:4]

Q_std_per_state = [0.01, 0.01, 0.01]  # Process noise: x1, x2, x3 (reduced for Lorenz)
R_std_per_state = [0.01, 0.01, 0.01]  # Measurement noise: x1, x2, x3 (reduced for Lorenz)

ess_threshold = n_particles * (opt_PF[6] if len(opt_PF) > 6 else opt_PF[4])  # Handle both 2D and 3D parameter arrays

pf_trainer = PF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    n_particles=n_particles,
    initial_weights=common_initial_weights,
    Q_std=Q_std_per_state, R_std=R_std_per_state,
    ess_threshold=ess_threshold
)
# Force identical particle initialization
for i in range(num_neurons):
    pf_trainer.particles[i] = np.tile(common_initial_weights[i], (pf_trainer.n_particles,1))
    pf_trainer.weights_pf[i] = np.ones(pf_trainer.n_particles)/pf_trainer.n_particles

x_hat_pf = np.zeros((n_steps, 3))
x_hat_pf[0] = x_true[0]

# --- RBPF --- (n_particles fixed at 100, with linear/nonlinear partitioning)
n_particles_rbpf = 50
rbpf_trainer = RBPF_RHONN_Trainer(
    num_neurons, num_weights_per_neuron,
    n_particles=n_particles_rbpf,
    initial_weights=common_initial_weights,
    linear_dims_ratio=opt_RBPF[7] if len(opt_RBPF) > 7 else (opt_RBPF[5] if len(opt_RBPF) > 5 else 0.4),
    Q_std=opt_RBPF[0:3] if len(opt_RBPF) > 6 else [opt_RBPF[0], opt_RBPF[1], 0.01], 
    R_std=opt_RBPF[3:6] if len(opt_RBPF) > 6 else [opt_RBPF[2], opt_RBPF[3], 0.01],
    ess_threshold=n_particles_rbpf * (opt_RBPF[6] if len(opt_RBPF) > 6 else opt_RBPF[4])
)
x_hat_rbpf = np.zeros((n_steps, 3))
x_hat_rbpf[0] = x_true[0]

# Initialize timing variables for each filter
import time
training_times = {'EKF': 0.0, 'UKF': 0.0, 'PF': 0.0, 'RBPF': 0.0}

print("\nStarting Lorenz system simulation (optimized params)...")
simulation_start_time = time.time()

for k in range(n_steps - 1):
    t_current = k * dt
    u_current = generate_realistic_trajectory(t_current, trajectory_type)
    x_true[k+1] = plant(x_true[k], u_current, dt, process_noise_type, process_noise_std, terrain_roughness, sensor_bias)

    # EKF - Measure training time
    ekf_start = time.time()
    ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ekf[k], x_hat_previous=x_hat_ekf[k], u_input=u_current)
    x_state_for_z_ekf = np.copy(x_hat_ekf[k])
    x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0], u_current)
    x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1], u_current)
    x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2], u_current)
    training_times['EKF'] += time.time() - ekf_start

    # UKF - Measure training time
    ukf_start = time.time()
    ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_ukf[k], x_hat_previous=x_hat_ukf[k], u_input=u_current)
    x_state_for_z_ukf = np.copy(x_hat_ukf[k])
    x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0], u_current)
    x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1], u_current)
    x_hat_ukf[k+1, 2] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[2], u_current)
    training_times['UKF'] += time.time() - ukf_start
    
    # PF - Measure training time
    pf_start = time.time()
    pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_pf[k], x_hat_previous=x_hat_pf[k], u_input=u_current)
    pf_weight_estimates = pf_trainer.get_estimate()
    x_state_for_z_pf = np.copy(x_hat_pf[k])
    x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0], u_current)
    x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1], u_current)
    x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[2], u_current)
    training_times['PF'] += time.time() - pf_start
    
    # RBPF - Measure training time
    rbpf_start = time.time()
    rbpf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_hat_rbpf[k], x_hat_previous=x_hat_rbpf[k], u_input=u_current)
    rbpf_weight_estimates = rbpf_trainer.get_estimate()
    x_state_for_z_rbpf = np.copy(x_hat_rbpf[k])
    x_hat_rbpf[k+1, 0] = RHONN_predict(x_state_for_z_rbpf, rbpf_weight_estimates[0], u_current)
    x_hat_rbpf[k+1, 1] = RHONN_predict(x_state_for_z_rbpf, rbpf_weight_estimates[1], u_current)
    x_hat_rbpf[k+1, 2] = RHONN_predict(x_state_for_z_rbpf, rbpf_weight_estimates[2], u_current)
    training_times['RBPF'] += time.time() - rbpf_start

    if k % (n_steps // 10) == 0:
        print(f"Simulation progress: {k/n_steps*100:.1f}%")

total_simulation_time = time.time() - simulation_start_time
print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [-0.00953486 -0.05338984  0.48592114  0.37844633  0.38883652  0.43415039
 -0.11044425 -0.13933742 -0.17608664  0.2457857  -0.32873624]
  Neuron 1: [ 0.46427098 -0.09409987 -0.0855379  -0.0393308  -0.16874781 -0.37853167
 -0.10010741 -0.29842054 -0.09539656 -0.39928821  0.01345027]
  Neuron 2: [ 0.02777427 -0.34891223  0.42925614  0.24141916  0.05249823 -0.22417203
  0.40660041  0.32393749  0.49430187  0.27976813 -0.3614566 ]

Using parameter sets:
EKF  -> Q_init=1.000e-05 R_init=1.000e-03 P_init=0.100 eta=0.100
UKF  -> Q_init=1.000e-05 R_init=1.000e-03 P_init=0.100 eta=0.100 alpha=1.000e-03
PF   -> Q=[0.010,0.010] R=[0.010,0.010] ESS_ratio=0.01
RBPF -> Q=[0.010,0.010] R=[0.010,0.010] ESS_ratio=0.01 Linear_ratio=0.01
RBPF Configuration:
  Linear dims (3): [8, 9, 10]
  Nonlinear dims (8): [0, 1, 2, 3, 4, 5, 6, 7]

Starting Lorenz system simulation (optimized params)...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 10.0%
Simu

In [11]:
# ============================================================
# 6) Results & plots for Lorenz Chaotic System
# ============================================================

mse_x1_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
mse_x2_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
mse_x3_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)

mse_x1_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
mse_x2_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
mse_x3_ukf = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2)

mse_x1_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
mse_x2_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)
mse_x3_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)

mse_x1_rbpf = np.mean((x_true[:, 0] - x_hat_rbpf[:, 0])**2)
mse_x2_rbpf = np.mean((x_true[:, 1] - x_hat_rbpf[:, 1])**2)
mse_x3_rbpf = np.mean((x_true[:, 2] - x_hat_rbpf[:, 2])**2)

mse_total_ekf = mse_x1_ekf + mse_x2_ekf + mse_x3_ekf
mse_total_ukf = mse_x1_ukf + mse_x2_ukf + mse_x3_ukf
mse_total_pf = mse_x1_pf + mse_x2_pf + mse_x3_pf
mse_total_rbpf = mse_x1_rbpf + mse_x2_rbpf + mse_x3_rbpf
mse_totals = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf, 'RBPF': mse_total_rbpf}
best_filter = min(mse_totals, key=mse_totals.get)


print("🎯" + "="*65)
print(f"🏆 MEJOR FILTRO: {best_filter} (MSE total: {mse_totals[best_filter]:.6f})")
print(f"🎲 SEMILLA USADA: {RANDOM_SEED}")
print("="*67)


print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    state_names = ['x1', 'x2', 'x3']
    print(f"  Neuron {i+1} ({state_names[i]}): {ekf_trainer.weights[i]}")

print(f"\nFinal UKF-RHONN Weights:")
for i in range(3):
    state_names = ['x1', 'x2', 'x3']
    print(f"  Neuron {i+1} ({state_names[i]}): {ukf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    state_names = ['x1', 'x2', 'x3']
    print(f"  Neuron {i+1} ({state_names[i]}): {pf_estimates[i]}")

print(f"\nFinal RBPF-RHONN Weight Estimates:")
rbpf_estimates = rbpf_trainer.get_estimate()
for i in range(3):
    state_names = ['x1', 'x2', 'x3']
    print(f"  Neuron {i+1} ({state_names[i]}): {rbpf_estimates[i]}")

print("\n--- Performance Comparison (MSE) ---")
# Calculate MSE for Lorenz system states
mse_x1_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)
mse_x2_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)
mse_x3_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)

mse_x1_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)
mse_x2_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)
mse_x3_ukf = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2)

mse_x1_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)
mse_x2_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)
mse_x3_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)

mse_x1_rbpf = np.mean((x_true[:, 0] - x_hat_rbpf[:, 0])**2)
mse_x2_rbpf = np.mean((x_true[:, 1] - x_hat_rbpf[:, 1])**2)
mse_x3_rbpf = np.mean((x_true[:, 2] - x_hat_rbpf[:, 2])**2)

mse_total_ekf = mse_x1_ekf + mse_x2_ekf + mse_x3_ekf
mse_total_ukf = mse_x1_ukf + mse_x2_ukf + mse_x3_ukf
mse_total_pf = mse_x1_pf + mse_x2_pf + mse_x3_pf
mse_total_rbpf = mse_x1_rbpf + mse_x2_rbpf + mse_x3_rbpf

print(f"EKF  MSE x1: {mse_x1_ekf:.6f}")
print(f"EKF  MSE x2: {mse_x2_ekf:.6f}")
print(f"EKF  MSE x3: {mse_x3_ekf:.6f}")
print(f"UKF  MSE x1: {mse_x1_ukf:.6f}")
print(f"UKF  MSE x2: {mse_x2_ukf:.6f}")
print(f"UKF  MSE x3: {mse_x3_ukf:.6f}")
print(f"PF   MSE x1: {mse_x1_pf:.6f}")
print(f"PF   MSE x2: {mse_x2_pf:.6f}")
print(f"PF   MSE x3: {mse_x3_pf:.6f}")
print(f"RBPF MSE x1: {mse_x1_rbpf:.6f}")
print(f"RBPF MSE x2: {mse_x2_rbpf:.6f}")
print(f"RBPF MSE x3: {mse_x3_rbpf:.6f}")

print("\n--- Total MSE ---")
print(f"EKF  Total MSE: {mse_total_ekf:.6f}")
print(f"UKF  Total MSE: {mse_total_ukf:.6f}")
print(f"PF   Total MSE: {mse_total_pf:.6f}")
print(f"RBPF Total MSE: {mse_total_rbpf:.6f}")

# Find best performing filter
mse_totals = {'EKF': mse_total_ekf, 'UKF': mse_total_ukf, 'PF': mse_total_pf, 'RBPF': mse_total_rbpf}
best_filter = min(mse_totals, key=mse_totals.get)

states_info = [
    {'idx': 0, 'var': 'x1', 'desc': 'First State', 'y_label': 'x1', 'chi': 'χ₁ (True x₁)', 'x': 'x₁ (Est.)'},
    {'idx': 1, 'var': 'x2', 'desc': 'Second State', 'y_label': 'x2', 'chi': 'χ₂ (True x₂)', 'x': 'x₂ (Est.)'},
    {'idx': 2, 'var': 'x3', 'desc': 'Third State', 'y_label': 'x3', 'chi': 'χ₃ (True x₃)', 'x': 'x₃ (Est.)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines', name=state_info['chi'], line=dict(color='black', width=2))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines', name=f"{state_info['x']} (EKF)", line=dict(dash='dash', color='blue'))
    trace_ukf = go.Scatter(x=t_history, y=x_hat_ukf[:, i], mode='lines', name=f"{state_info['x']} (UKF)", line=dict(dash='dashdot', color='green'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines', name=f"{state_info['x']} (PF)", line=dict(dash='dot', color='red'))
    trace_rbpf = go.Scatter(x=t_history, y=x_hat_rbpf[:, i], mode='lines', name=f"{state_info['x']} (RBPF)", line=dict(dash='longdash', color='purple'))
    fig = go.Figure([trace_plant, trace_ekf, trace_ukf, trace_pf, trace_rbpf])
    fig.update_layout(title=f'Lorenz System RHONN Identification - {state_info["var"]}', xaxis_title='Time (s)', yaxis_title=state_info['y_label'], legend=dict(x=0, y=1, orientation='h'), font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white')
    fig.show()

error_x1_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_x2_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_x3_ekf = x_true[:, 2] - x_hat_ekf[:, 2]
error_x1_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_x2_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
error_x3_ukf = x_true[:, 2] - x_hat_ukf[:, 2]
error_x1_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_x2_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_x3_pf = x_true[:, 2] - x_hat_pf[:, 2]
error_x1_rbpf = x_true[:, 0] - x_hat_rbpf[:, 0]
error_x2_rbpf = x_true[:, 1] - x_hat_rbpf[:, 1]
error_x3_rbpf = x_true[:, 2] - x_hat_rbpf[:, 2]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_ekf, mode='lines', name=f'EKF Err x₁ ({mse_x1_ekf:.2e})', opacity=0.7, line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_ukf, mode='lines', name=f'UKF Err x₁ ({mse_x1_ukf:.2e})', opacity=0.7, line=dict(color='green')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_pf, mode='lines', name=f'PF Err x₁ ({mse_x1_pf:.2e})', opacity=0.7, line=dict(color='red')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_rbpf, mode='lines', name=f'RBPF Err x₁ ({mse_x1_rbpf:.2e})', opacity=0.7, line=dict(color='purple')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_ekf, mode='lines', name=f'EKF Err x₂ ({mse_x2_ekf:.2e})', opacity=0.7, line=dict(color='blue', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_ukf, mode='lines', name=f'UKF Err x₂ ({mse_x2_ukf:.2e})', opacity=0.7, line=dict(color='green', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_pf, mode='lines', name=f'PF Err x₂ ({mse_x2_pf:.2e})', opacity=0.7, line=dict(color='red', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_rbpf, mode='lines', name=f'RBPF Err x₂ ({mse_x2_rbpf:.2e})', opacity=0.7, line=dict(color='purple', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x3_ekf, mode='lines', name=f'EKF Err x₃ ({mse_x3_ekf:.2e})', opacity=0.7, line=dict(color='blue', dash='longdash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x3_ukf, mode='lines', name=f'UKF Err x₃ ({mse_x3_ukf:.2e})', opacity=0.7, line=dict(color='green', dash='longdash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x3_pf, mode='lines', name=f'PF Err x₃ ({mse_x3_pf:.2e})', opacity=0.7, line=dict(color='red', dash='longdash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x3_rbpf, mode='lines', name=f'RBPF Err x₃ ({mse_x3_rbpf:.2e})', opacity=0.7, line=dict(color='purple', dash='longdash')))
fig2.update_layout(title='Identification Errors (MSE values)', xaxis_title='Time (s)', yaxis_title='Error', legend=dict(x=0, y=1, orientation='h'), font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white')
fig2.show()

# 3D Lorenz Attractor Plot
fig_3d = go.Figure()
fig_3d.add_trace(go.Scatter3d(x=x_true[:, 0], y=x_true[:, 1], z=x_true[:, 2], mode='lines', name='True Lorenz Attractor', line=dict(color='black', width=4)))
fig_3d.add_trace(go.Scatter3d(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], z=x_hat_ekf[:, 2], mode='lines', name='EKF Estimate', line=dict(color='blue', width=3)))
fig_3d.add_trace(go.Scatter3d(x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1], z=x_hat_ukf[:, 2], mode='lines', name='UKF Estimate', line=dict(color='green', width=3)))
fig_3d.add_trace(go.Scatter3d(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], z=x_hat_pf[:, 2], mode='lines', name='PF Estimate', line=dict(color='red', width=3)))
fig_3d.add_trace(go.Scatter3d(x=x_hat_rbpf[:, 0], y=x_hat_rbpf[:, 1], z=x_hat_rbpf[:, 2], mode='lines', name='RBPF Estimate', line=dict(color='purple', width=3)))
fig_3d.add_trace(go.Scatter3d(x=[x_true[0, 0]], y=[x_true[0, 1]], z=[x_true[0, 2]], mode='markers', name='Start', marker=dict(color='green', size=8, symbol='diamond')))
fig_3d.add_trace(go.Scatter3d(x=[x_true[-1, 0]], y=[x_true[-1, 1]], z=[x_true[-1, 2]], mode='markers', name='End', marker=dict(color='red', size=8, symbol='square')))
fig_3d.update_layout(title='Lorenz Attractor - 3D Phase Space Comparison', scene=dict(xaxis_title='x₁', yaxis_title='x₂', zaxis_title='x₃'), font=dict(size=12), showlegend=True)
fig_3d.show()

# 2D projections of Lorenz attractor
fig_proj = go.Figure()
fig_proj.add_trace(go.Scatter(x=x_true[:, 0], y=x_true[:, 1], mode='lines', name='True (x₁ vs x₂)', line=dict(color='black', width=3)))
fig_proj.add_trace(go.Scatter(x=x_hat_rbpf[:, 0], y=x_hat_rbpf[:, 1], mode='lines', name=f'RBPF (x₁ vs x₂)', line=dict(color='purple', width=2, dash='dash')))
fig_proj.add_trace(go.Scatter(x=[x_true[0, 0]], y=[x_true[0, 1]], mode='markers', name='Start', marker=dict(color='green', size=10, symbol='star')))
fig_proj.add_trace(go.Scatter(x=[x_true[-1, 0]], y=[x_true[-1, 1]], mode='markers', name='End', marker=dict(color='red', size=10, symbol='square')))
fig_proj.update_layout(title='Lorenz Attractor Projection (x₁ vs x₂) - Best Filter (RBPF)', xaxis_title='x₁', yaxis_title='x₂', font=dict(size=12), plot_bgcolor='white', paper_bgcolor='white', showlegend=True)
fig_proj.show()


# === RESUMEN DE RENDIMIENTO CON SEMILLA ===
print("\n📊 MSE Desglosado por Filtro:")
print(f"   EKF: {mse_total_ekf:.6f}  |  UKF: {mse_total_ukf:.6f}  |  PF: {mse_total_pf:.6f}  |  RBPF: {mse_total_rbpf:.6f}")

print("\n⏱️ Tiempo de Entrenamiento por Filtro:")
print(f"   EKF: {training_times['EKF']:.4f}s  |  UKF: {training_times['UKF']:.4f}s  |  PF: {training_times['PF']:.4f}s  |  RBPF: {training_times['RBPF']:.4f}s")
print(f"   Total Simulación: {total_simulation_time:.4f}s")

# Calculate relative computational efficiency
fastest_filter = min(training_times, key=training_times.get)
efficiency_ratios = {name: training_times[name]/training_times[fastest_filter] for name in training_times.keys()}
print("\n🚀 Eficiencia Computacional (relativo al más rápido):")
for name, ratio in efficiency_ratios.items():
    efficiency_pct = 100/ratio if ratio > 0 else 0
    print(f"   {name}: {ratio:.2f}x slower ({efficiency_pct:.1f}% efficiency)")

print(f"\n💡 Para reproducir estos resultados:")
print(f"   Principal: RANDOM_SEED = {RANDOM_SEED}")
print(f"   DE Optim.: DE_SEED = {(RANDOM_SEED + 12345) % 100000}")
print(f"   (Cambia línea 8 en celda 4 para usar semilla principal)")

# --- Parameter summary ---
print("\n--- Optimized Parameter Summary ---")
print(f"EKF params: Q={ekf_trainer.Q[0][0,0]:.3e} R={ekf_trainer.R[0][0]:.3e} P0~{ekf_trainer.P[0][0,0]:.3e} eta={ekf_trainer.eta:.3f}")
print(f"UKF params: alpha={ukf_trainer.alpha:.3e} eta={ukf_trainer.eta:.3f} Qdiag={ukf_trainer.Q[0][0,0]:.3e} R={ukf_trainer.R[0][0]:.3e}")
print(f"PF params: Q_std={pf_trainer.Q_std} R_std={pf_trainer.R_std} ESS_th={pf_trainer.ess_threshold:.1f} n_particles={pf_trainer.n_particles}")
print(f"RBPF params: Q_std={rbpf_trainer.Q_std} R_std={rbpf_trainer.R_std} ESS_th={rbpf_trainer.ess_threshold:.1f} n_particles={rbpf_trainer.n_particles} linear_dims={rbpf_trainer.n_linear}/{rbpf_trainer.num_weights_per_neuron}")

print("\nOptimization + simulation complete.")

# 58865

🎯=================================================================
🏆 MEJOR FILTRO: RBPF (MSE total: 0.356863)
🎲 SEMILLA USADA: 10932

Final EKF-RHONN Weights:
  Neuron 1 (x1): [-0.27250081 -0.28332818  0.84271546 -0.14310634  0.09514739  0.14033894
 -0.10961552 -0.04643767 -0.38741346  1.57446452 -0.18007889]
  Neuron 2 (x2): [ 0.40819067  0.49746109 -0.22097729 -0.01506387 -0.13100529 -0.10150246
  0.03078676 -0.04423764 -0.15653408  0.28360562  0.02902639]
  Neuron 3 (x3): [-0.20679201 -0.34564672  0.73116519 -0.0763461   0.00170617 -0.04569856
  0.18080821 -0.11841559  0.04991006  1.36359444 -0.20807534]

Final UKF-RHONN Weights:
  Neuron 1 (x1): [-0.82848856  0.26815187  0.21192941 -0.025061    0.2267946   0.07317604
  0.00828353 -0.10386332 -0.09277489  0.53555641 -0.04406336]
  Neuron 2 (x2): [-0.21709271  0.86017879 -0.52416176 -0.03385186  0.09437834 -0.19697533
  0.26173089 -0.10670953 -0.08372733  0.13753147  0.43676042]
  Neuron 3 (x3): [ 0.41253767  0.89168073  0.01837054  


📊 MSE Desglosado por Filtro:
   EKF: 26.353974  |  UKF: 9.990039  |  PF: 3.868073  |  RBPF: 0.356863

⏱️ Tiempo de Entrenamiento por Filtro:
   EKF: 0.2589s  |  UKF: 0.7863s  |  PF: 0.5905s  |  RBPF: 9.1875s
   Total Simulación: 10.8522s

🚀 Eficiencia Computacional (relativo al más rápido):
   EKF: 1.00x slower (100.0% efficiency)
   UKF: 3.04x slower (32.9% efficiency)
   PF: 2.28x slower (43.8% efficiency)
   RBPF: 35.48x slower (2.8% efficiency)

💡 Para reproducir estos resultados:
   Principal: RANDOM_SEED = 10932
   DE Optim.: DE_SEED = 23277
   (Cambia línea 8 en celda 4 para usar semilla principal)

--- Optimized Parameter Summary ---
EKF params: Q=1.000e-05 R=1.000e-03 P0~1.163e-02 eta=0.100
UKF params: alpha=1.000e-03 eta=0.100 Qdiag=1.000e-05 R=1.000e-03
PF params: Q_std=[0.01, 0.01, 0.01] R_std=[0.01, 0.01, 0.01] ESS_th=100.0 n_particles=200
RBPF params: Q_std=[0.01, 0.01, 0.01] R_std=[0.01, 0.01, 0.01] ESS_th=32.5 n_particles=50 linear_dims=3/11

Optimization + simulation 

In [12]:
# ============================================================
# Training Time Comparison Visualization
# ============================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Training time comparison bar chart
filter_names = list(training_times.keys())
times = list(training_times.values())
colors = ['#1f77b4', '#2ca02c', '#d62728', '#9467bd']  # Blue, Green, Red, Purple

fig_timing = go.Figure()
fig_timing.add_trace(go.Bar(
    x=filter_names,
    y=times,
    marker=dict(color=colors),
    text=[f'{t:.4f}s' for t in times],
    textposition='auto',
    name='Training Time'
))

fig_timing.update_layout(
    title='Training Time Comparison by Filter Type',
    xaxis_title='Filter Type',
    yaxis_title='Training Time (seconds)',
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False
)

fig_timing.show()

# Combined Performance vs Speed Analysis
fig_combined = make_subplots(
    rows=1, cols=2,
    subplot_titles=('MSE Performance (lower is better)', 'Training Time (lower is better)'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}]]
)

# MSE subplot
mse_values = [mse_total_ekf, mse_total_ukf, mse_total_pf, mse_total_rbpf]
fig_combined.add_trace(
    go.Bar(x=filter_names, y=mse_values, marker=dict(color=colors), 
           text=[f'{mse:.2e}' for mse in mse_values], textposition='auto',
           name='MSE'),
    row=1, col=1
)

# Training time subplot  
fig_combined.add_trace(
    go.Bar(x=filter_names, y=times, marker=dict(color=colors),
           text=[f'{t:.3f}s' for t in times], textposition='auto',
           name='Time'),
    row=1, col=2
)

fig_combined.update_layout(
    title_text="Performance vs Computational Efficiency Trade-off Analysis",
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=False
)

fig_combined.update_xaxes(title_text="Filter Type", row=1, col=1)
fig_combined.update_xaxes(title_text="Filter Type", row=1, col=2)
fig_combined.update_yaxes(title_text="MSE (Total)", row=1, col=1)
fig_combined.update_yaxes(title_text="Training Time (s)", row=1, col=2)

fig_combined.show()

# Performance efficiency metric: accuracy per computation time
print("\n🎯 Performance-Efficiency Trade-off Analysis:")
print("   (Lower MSE/Time ratio = better overall efficiency)")
for i, name in enumerate(filter_names):
    efficiency_metric = mse_values[i] / times[i] if times[i] > 0 else float('inf')
    print(f"   {name}: MSE/Time = {efficiency_metric:.2e}")
    
# Rank filters by different criteria
mse_ranking = sorted(zip(filter_names, mse_values), key=lambda x: x[1])
time_ranking = sorted(zip(filter_names, times), key=lambda x: x[1])
efficiency_ranking = sorted(zip(filter_names, [mse_values[i]/times[i] for i in range(len(filter_names))]), key=lambda x: x[1])

print("\n🏆 Rankings:")
print("   By Accuracy (MSE):", [name for name, _ in mse_ranking])
print("   By Speed (Time):", [name for name, _ in time_ranking])
print("   By Overall Efficiency (MSE/Time):", [name for name, _ in efficiency_ranking])


🎯 Performance-Efficiency Trade-off Analysis:
   (Lower MSE/Time ratio = better overall efficiency)
   EKF: MSE/Time = 1.02e+02
   UKF: MSE/Time = 1.27e+01
   PF: MSE/Time = 6.55e+00
   RBPF: MSE/Time = 3.88e-02

🏆 Rankings:
   By Accuracy (MSE): ['RBPF', 'PF', 'UKF', 'EKF']
   By Speed (Time): ['EKF', 'PF', 'UKF', 'RBPF']
   By Overall Efficiency (MSE/Time): ['RBPF', 'PF', 'UKF', 'EKF']
